# Training MobileNetV2 — Klasifikasi Fertilitas Telur (Fertile vs Infertile)

Notebook **khusus classifier MobileNetV2** (tanpa training YOLO). Dataset Roboflow **di-download langsung di notebook ini** (config sama persis dgn v3), lalu diubah menjadi **crop GT** siap `ImageFolder` + training optimal.

```
Roboflow YOLO (3 dataset) -> pool + remap -> audit duplikat -> split group-aware -> crop GT -> (+ data sendiri) -> MobileNetV2
```

## Dataset bawaan (sama dgn v3)

- `duck-egg-detection-roboflow` (samsuri-2ug2l, v2) : fertil, infertil
- `egg-fertility-candling-roboflow` (nodkeyobma, v9) : fertile, infertile
- `deteksi-fertilitas-telur-2` (misnis-workspace-z1d0y, v1) : fertile, infertile
- `egg-fertility-nitkl` (nonaktif default, tinggal uncomment)

Normalisasi otomatis: `fertil->fertile`, `infertil->infertile` (case-insensitive), anti-terbalik via nama kelas.
Data sendiri tetap bisa ditambah via `CUSTOM_DIRS` / `CUSTOM_ZIPS` (digabung + dedup MD5).

## Urutan jalankan

1. Sel instalasi sekali -> **Restart session** -> lewati sel instalasi.
2. Jalankan berurutan Sel 2 -> akhir.

Output: `mobilenet_egg_best.pt`, `classifier.onnx`, `classifier_int8.onnx`, `inference_config.json` (+ `merged_manifest.csv`).


## 1. Instalasi dependensi (jalankan sekali, lalu RESTART)

Jangan gabung `pip install` + `import torch` dalam satu sesi — penyebab umum error `AttributeError: module 'torch' has no attribute '_utils'`.

In [ ]:
# SEL INSTALASI — jalankan sekali, lalu: Runtime -> Restart session.
# Setelah restart, LEWATI sel ini.
%pip install -q ultralytics roboflow kagglehub pyyaml imagehash scikit-learn pandas seaborn pillow onnx onnxruntime
print("Instalasi selesai.")
print("WAJIB: Runtime -> Restart session, lalu JANGAN jalankan sel ini lagi.")


In [ ]:
import torch
print("=" * 50)
print(f"torch: {torch.__version__}")
try:
    import torch._utils
    print("torch._utils OK")
except Exception as e:
    print(f"TORCH RUSAK ({e}). Solusi: Runtime -> Restart session.")
print("=" * 50)
if torch.cuda.is_available():
    print(f"GPU Aktif  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("CPU mode (tetap bisa training, lebih lambat).")
print("=" * 50)

## 2. Mount Google Drive (untuk backup model + restore dataset)

Fallback ke `/content/egg_models` bila tanpa Drive.

In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MODEL_DIR = Path("/content/drive/MyDrive/egg_fertility_models")
    DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Drive terhubung: {DRIVE_MODEL_DIR}")
except Exception as e:
    print(f"Tanpa Drive ({e}), pakai folder lokal.")
    DRIVE_MODEL_DIR = Path("/content/egg_models")
    DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Folder lokal: {DRIVE_MODEL_DIR}")

EXPORT_DIR = DRIVE_MODEL_DIR / "models_exported"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Konfigurasi training (optimal untuk MobileNetV2)

Default: input 320px (tekstur pembuluh darah lebih jelas), AdamW, AMP, label smoothing, sampler seimbang, augmentasi kuat, early stopping berbasis F2-fertile.

In [ ]:
from pathlib import Path

# --- kompatibel dengan v3: JANGAN ubah urutan ini ---
if "TARGET_CLASSES" not in globals():
    TARGET_CLASSES = ["fertile", "infertile"]
if "SEED" not in globals():
    SEED = 42

# --- path dataset (format diselaraskan dgn v3) ---
DATASET_BASE_DIR = Path("/content/datasets")
POOL_DIR = Path("/content/egg_pool")
V3_FINAL_DEFAULT = Path("/content/egg_dataset_final")
V3_CROPS_DEFAULT = Path("/content/egg_crops")
CROPS_DIR = Path(globals().get("CROPS_DIR", V3_CROPS_DEFAULT))
MERGED_CROPS_DIR = Path("/content/egg_crops_merged")

# --- hyperparameter optimal ---
IMG_SIZE = 320
BATCH = 16
NUM_WORKERS = 2
EPOCHS_HEAD = 5
EPOCHS_FT = 30
LR_HEAD = 1e-3
LR_FT = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2
PATIENCE = 8
LABEL_SMOOTHING = 0.1
DROPOUT_P = 0.2
USE_SAMPLER = True
USE_CLASS_WEIGHTS = False  # dinonaktifkan agar tidak double-weighting dengan sampler
USE_MIXUP = False
MIXUP_ALPHA = 0.2
AMP = True
BETA_FERTILE = 1.0  # F1 seimbang (meredam false positive telur merah)
CROP_MARGIN = 0.10

print(f"TARGET_CLASSES : {TARGET_CLASSES}")
print(f"CROPS_DIR      : {CROPS_DIR}")
print(f"MERGED_DIR     : {MERGED_CROPS_DIR}")
print(f"IMG {IMG_SIZE} | BATCH {BATCH} | SEED {SEED} | AMP {AMP}")


## 4. Download dataset Roboflow (config sama dgn v3)

DatasetManager robust: `overwrite=True`, flatten nested, verifikasi jumlah gambar. API key dari Colab Secrets / env / getpass.


In [ ]:
import os, shutil
from pathlib import Path
from roboflow import Roboflow
def _count_images(root: Path) -> int:
    exts = {'.jpg','.jpeg','.png','.bmp','.webp'}; n=0
    for sub in ['train','valid','test','val']:
        d=root/sub/'images'
        q=sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in exts) if d.exists() else 0
        n+=q
    d=root/'images'
    q=sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in exts) if (n==0 and d.exists()) else 0
    return n+q
def _find_dataset_root(base: Path) -> Path:
    if not base.exists(): return base
    subs=['train','valid','test','val']
    for s in subs:
        if (base/s/'images').exists(): return base
    import glob as _g
    if list(base.glob('*.yaml')) or list(base.glob('*.yml')): return base
    subdirs=[d for d in base.iterdir() if d.is_dir()]
    if len(subdirs)==1:
        inner=subdirs[0]
        for s in subs:
            if (inner/s/'images').exists(): return inner
        if list(inner.glob('*.yaml')): return inner
    for cand in base.rglob('train/images'):
        if cand.is_dir(): return cand.parent.parent
    return base
def _flatten_if_nested(target_dir: Path) -> Path:
    root=_find_dataset_root(target_dir)
    if root!=target_dir:
        print(f"  nested ({root.relative_to(target_dir)}), meratakan...")
        for item in list(root.iterdir()):
            dest=target_dir/item.name
            exists=dest.exists()
            isdir=dest.is_dir() if exists else False
            if exists:
                if isdir: shutil.rmtree(dest)
                else: dest.unlink()
            shutil.move(str(item),str(dest))
        try: shutil.rmtree(root)
        except Exception: pass
    return target_dir
class DatasetManager:
    def __init__(self, base_save_dir='/content/datasets'):
        self.base_dir=Path(base_save_dir); self.base_dir.mkdir(parents=True,exist_ok=True)
    def _download_roboflow(self, config, target_dir):
        print(f"DL [{config['name']}] {config['workspace']}/{config['project']} v{config['version']}")
        key=config.get('api_key') or ''
        if not key: raise RuntimeError('ROBOFLOW_API_KEY kosong')
        if target_dir.exists(): shutil.rmtree(target_dir)
        rf=Roboflow(api_key=key)
        ver=rf.workspace(config['workspace']).project(config['project']).version(config['version'])
        try: ds=ver.download(model_format=config['format'],location=str(target_dir),overwrite=True)
        except TypeError: ds=ver.download(config['format'],location=str(target_dir))
        _flatten_if_nested(target_dir)
        n=_count_images(target_dir); print(f"  gambar: {n}")
        if n==0: raise RuntimeError('0 gambar, cek universe.roboflow.com')
    def download_dataset(self, config):
        td=self.base_dir/config['name']
        ex=td.exists(); cnt=_count_images(td) if ex else 0
        if ex and cnt>0: print(f"[{config['name']}] sudah ada ({cnt}), dilewati."); return td
        if ex: shutil.rmtree(td)
        self._download_roboflow(config,td); return td
def download_all(manager, cfgs):
    res={}
    for c in cfgs:
        try: pt=manager.download_dataset(c); res[c['name']]={'status':'success','path':str(pt),'n_images':_count_images(Path(pt))}
        except Exception as e: res[c['name']]={'status':'failed','error':str(e)}; print(f"Gagal {c['name']}: {e}")
    return res


In [ ]:
import getpass, os
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY=userdata.get('ROBOFLOW_API_KEY')
except Exception: ROBOFLOW_API_KEY=os.getenv('ROBOFLOW_API_KEY')
if not ROBOFLOW_API_KEY: ROBOFLOW_API_KEY=getpass.getpass('Roboflow API Key: ')
manager=DatasetManager(base_save_dir=DATASET_BASE_DIR)
DATASETS_CONFIG=[
    {'name':'duck-egg-detection-roboflow','source_type':'roboflow','api_key':ROBOFLOW_API_KEY,'workspace':'samsuri-2ug2l','project':'duck-egg-detection','version':2,'format':'yolov8','expected_names':['fertil','infertil']},
    {'name':'egg-fertility-candling-roboflow','source_type':'roboflow','api_key':ROBOFLOW_API_KEY,'workspace':'nodkeyobma','project':'egg-fertility-candling','version':9,'format':'yolov8','expected_names':['fertile','infertile']},
    {'name':'deteksi-fertilitas-telur-2','source_type':'roboflow','api_key':ROBOFLOW_API_KEY,'workspace':'misnis-workspace-z1d0y','project':'deteksi-fertilitas-telur-2','version':1,'format':'yolov8','expected_names':['fertile','infertile']},
]
# # aktifkan bila perlu:
# DATASETS_CONFIG.append({'name':'egg-fertility-nitkl','source_type':'roboflow','api_key':ROBOFLOW_API_KEY,'workspace':'electronic-engineering-polytechnic-institute-of-surabaya-zxfjf','project':'egg-fertility-nitkl','version':9,'format':'yolov8','expected_names':['Fertile','Infertile']})
results=download_all(manager,DATASETS_CONFIG)
print('='*50)
ok=True
for k,v in results.items():
    tag='SUKSES' if v['status']=='success' else 'GAGAL '
    extra=f" ({v.get('n_images',0)} gambar)" if v['status']=='success' else ''
    print(f"{tag} - {k}: {v.get('path',v.get('error'))}{extra}")
    ok=ok and (v['status']=='success')
print('='*50)
bad=[k for k,v in results.items() if v['status']!='success']
warn_empty=[k for k,v in results.items() if v['status']=='success' and v.get('n_images',0)==0]
cond_all_fail=len(bad)==len(results)
cond_empty=len(warn_empty)>0
if cond_all_fail: raise RuntimeError('Semua download GAGAL.')
if cond_empty: print(f"PERINGATAN: {warn_empty} kosong.")


## 5. Pool + remap + audit duplikat + split group-aware (format v3, optimal classifier)

Gambar YOLO ditumpuk jadi pool, dinormalisasi (`fertil->fertile`), diaudit base-ID + pHash (union-find), split 80/10/10 group-aware + stratified. Output: `FINAL_DIR` + `data.yaml` + `metadata.csv`.


In [ ]:
import re, shutil, yaml, pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm
EXPLICIT_REMAP={'duck-egg-detection-roboflow':{0:0,1:1},'egg-fertility-candling-roboflow':{0:0,1:1},'egg-fertility-nitkl':{0:0,1:1},'deteksi-fertilitas-telur-2':{0:0,1:1}}
NAME_NORM={'fertil':'fertile','fertile':'fertile','infertil':'infertile','infertile':'infertile'}
VALID_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}
RF_SUFFIX=re.compile(r'[._-]?(jpe?g|png|bmp|webp)\.rf\.[0-9a-f]+$',re.IGNORECASE)
def base_id(stem: str) -> str: return RF_SUFFIX.sub('',stem).strip().lower()
def _resolve_root2(ds_path: Path) -> Path:
    q=ds_path.exists()
    s1=(ds_path/'train'/'images').exists() if q else False
    s2=(ds_path/'valid'/'images').exists() if q else False
    s3=(ds_path/'test'/'images').exists() if q else False
    s4=(ds_path/'val'/'images').exists() if q else False
    if s1 or s2 or s3 or s4: return ds_path
    return _find_dataset_root(ds_path)
def read_class_names(ds_path: Path):
    ds_path=_resolve_root2(ds_path)
    cands=sorted(list(ds_path.glob('*.yaml'))+list(ds_path.glob('*.yml')),key=lambda p:(p.name!='data.yaml',p.name))
    nocand=len(cands)==0
    if nocand: cands=list(ds_path.glob('*/*.yaml'))+list(ds_path.glob('*/*.yml'))
    for yf in cands:
        try:
            data=yaml.safe_load(open(yf)); names=data.get('names',[]) if isinstance(data,dict) else []
            isdict=isinstance(names,dict)
            if isdict: names=[names[k] for k in sorted(names.keys())]
            lst=list(names)
            if len(lst)>0: return lst
        except Exception: continue
    return []
def build_auto_remap(actual):
    remap={}
    for old_id,name in enumerate(actual):
        norm=NAME_NORM.get(str(name).strip().lower())
        bad=norm is None
        if bad: print(f"  kelas '{name}' dibuang."); continue
        remap[old_id]=TARGET_CLASSES.index(norm)
    return remap
def build_pool(datasets_config):
    ex=POOL_DIR.exists()
    if ex: shutil.rmtree(POOL_DIR)
    (POOL_DIR/'images').mkdir(parents=True,exist_ok=True); (POOL_DIR/'labels').mkdir(parents=True,exist_ok=True)
    rows=[]
    for idx,config in enumerate(datasets_config):
        name=config['name']; ds_path=_resolve_root2(DATASET_BASE_DIR/name)
        miss=not (DATASET_BASE_DIR/name).exists()
        if miss: print(f"LEWAT: {name} tidak ada."); continue
        actual=read_class_names(ds_path)
        noyaml=len(actual)==0
        if noyaml: print(f"LEWAT: [{name}] tanpa data.yaml."); continue
        remap=build_auto_remap(actual)
        exp=EXPLICIT_REMAP.get(name)
        same=exp is not None and set(exp.keys())==set(remap.keys()) and all(exp[k]==remap[k] for k in remap)
        if same: remap=exp
        print(f"[{name}] kelas={actual} -> remap={remap}")
        empty=len(remap)==0
        if empty: continue
        prefix=f"ds{idx+1}"; seen=set()
        for split in ['train','valid','test','val']:
            si=ds_path/split/'images'; sl=ds_path/split/'labels'
            noimg=not si.exists()
            if noimg: continue
            real=si.resolve(); dup=real in seen
            if dup: continue
            seen.add(real)
            imgs=[f for f in si.iterdir() if f.is_file() and f.suffix.lower() in VALID_EXTS]
            for img_p in tqdm(imgs,desc=f"{name} [{split}]",leave=False):
                new_stem=f"{prefix}_{img_p.stem}"; dst_img=POOL_DIR/'images'/f"{new_stem}{img_p.suffix.lower()}"
                shutil.copy2(img_p,dst_img)
                src_lbl=sl/f"{img_p.stem}.txt"; counts={0:0,1:0}; lines=[]
                hasex=src_lbl.exists()
                if hasex:
                    for line in src_lbl.read_text().splitlines():
                        parts=line.strip().split()
                        short=len(parts)<5
                        if short: continue
                        try: old=int(float(parts[0]))
                        except ValueError: continue
                        skip=old not in remap
                        if skip: continue
                        nc=remap[old]; counts[nc]+=1; lines.append(f"{nc} "+' '.join(parts[1:5]))
                (POOL_DIR/'labels'/f"{new_stem}.txt").write_text(chr(10).join(lines)+(chr(10) if lines else ''))
                rows.append({'file':dst_img.name,'stem':new_stem,'source':name,'source_idx':idx+1,'orig_split':split,'base_id':f"{prefix}_{base_id(img_p.stem)}",'n_fertile':counts[0],'n_infertile':counts[1],'n_boxes':counts[0]+counts[1]})
    norows=len(rows)==0
    if norows: raise RuntimeError('POOL KOSONG.')
    df=pd.DataFrame(rows)
    zero=(df['n_boxes']==0).sum()
    print(f"Pool: {len(df)} gambar | 0-box: {zero}")
    return df
pool_df=build_pool(DATASETS_CONFIG)
pool_df.groupby('source')[['n_fertile','n_infertile']].sum()


In [ ]:
from PIL import Image
import imagehash, numpy as np
from sklearn.model_selection import StratifiedGroupKFold
parent={}
def find(x):
    parent.setdefault(x,x)
    cond=parent[x]!=x
    while cond: parent[x]=parent[parent[x]]; x=parent[x]; cond=parent[x]!=x
    return x
def union(a,b):
    ra=find(a); rb=find(b)
    diff=ra!=rb
    if diff: parent[rb]=ra
for stem in pool_df['stem']: find(stem)
for _,g in pool_df.groupby('base_id'):
    ss=g['stem'].tolist()
    multi=len(ss)>1
    if multi:
        for s in ss[1:]: union(ss[0],s)
print('pHash (beberapa menit)...')
phash_map=defaultdict(list)
for _,row in tqdm(pool_df.iterrows(),total=len(pool_df)):
    try: h=str(imagehash.phash(Image.open(POOL_DIR/'images'/row['file']).convert('RGB')))
    except Exception: h='ERR_'+row['stem']
    phash_map[h].append(row['stem'])
for h,ss in phash_map.items():
    dup=len(ss)>1
    if dup:
        for s in ss[1:]: union(ss[0],s)
pool_df['group']=pool_df['stem'].map(find)
print(f"Grup unik: {pool_df['group'].nunique()} / {len(pool_df)}")
np.random.seed(SEED)
def dom(r):
    z=r['n_boxes']==0
    if z: return 'empty'
    f=r['n_fertile']>0; it=r['n_infertile']>0
    both=f and it
    if both: return 'mixed'
    return 'fertile' if f else 'infertile'
pool_df['label']=pool_df.apply(dom,axis=1)
pool_df['stratum']=pool_df['source_idx'].astype(str)+'_'+pool_df['label']
vc=pool_df['stratum'].value_counts(); rare=vc[vc<20].index
pool_df.loc[pool_df['stratum'].isin(rare),'stratum']='rare'
X=np.arange(len(pool_df)); y=pool_df['stratum'].values; groups=pool_df['group'].values
rest_idx,test_idx=next(StratifiedGroupKFold(n_splits=10,shuffle=True,random_state=SEED).split(X,y,groups))
tr_rel,val_rel=next(StratifiedGroupKFold(n_splits=9,shuffle=True,random_state=SEED).split(X[rest_idx],y[rest_idx],groups[rest_idx]))
train_idx,valid_idx=rest_idx[tr_rel],rest_idx[val_rel]
pool_df['split']=None
pool_df.iloc[train_idx,pool_df.columns.get_loc('split')]='train'
pool_df.iloc[valid_idx,pool_df.columns.get_loc('split')]='valid'
pool_df.iloc[test_idx,pool_df.columns.get_loc('split')]='test'
leak=(pool_df.groupby('group')['split'].nunique()==1).all()
assert leak,'MASIH BOCOR!'
print('Split OK, tanpa kebocoran grup.')
print(pool_df.groupby('split')['label'].value_counts().unstack(fill_value=0))
FINAL_DIR=V3_FINAL_DEFAULT
import os
fex=FINAL_DIR.exists()
if fex: shutil.rmtree(FINAL_DIR)
for sp in ['train','valid','test']:
    (FINAL_DIR/sp/'images').mkdir(parents=True,exist_ok=True); (FINAL_DIR/sp/'labels').mkdir(parents=True,exist_ok=True)
for _,row in tqdm(pool_df.iterrows(),total=len(pool_df),desc='Final'):
    si=POOL_DIR/'images'/row['file']; sl=POOL_DIR/'labels'/f"{row['stem']}.txt"
    di=FINAL_DIR/row['split']/'images'/row['file']; dl=FINAL_DIR/row['split']/'labels'/f"{row['stem']}.txt"
    try: os.link(si,di); os.link(sl,dl)
    except OSError: shutil.copy2(si,di); shutil.copy2(sl,dl)
yaml_path=FINAL_DIR/'data.yaml'
open(yaml_path,'w').write(f"path: {FINAL_DIR.resolve()}\ntrain: train/images\nval: valid/images\ntest: test/images\nnc: 2\nnames: ['fertile', 'infertile']\n")
pool_df.to_csv(FINAL_DIR/'metadata.csv',index=False)
print(yaml_path.read_text())


## 6. Build crop GT untuk MobileNet (dari box ground truth)

Setiap box GT di-crop + margin 10% menjadi ImageFolder `egg_crops/split/fertile|infertile`. Dari GT (bukan prediksi) agar classifier belajar dari box bersih.


In [ ]:
from PIL import Image
from tqdm.auto import tqdm
CROPS_BASE=V3_CROPS_DEFAULT
cex=CROPS_BASE.exists()
import shutil as _sh
if cex: _sh.rmtree(CROPS_BASE)
n_saved=0
for split in ['train','valid','test']:
    img_dir=FINAL_DIR/split/'images'; lbl_dir=FINAL_DIR/split/'labels'
    cand=[p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS]
    for img_p in tqdm(cand,desc=f"Crop [{split}]"):
        lp=lbl_dir/f"{img_p.stem}.txt"
        ok=lp.exists() and lp.stat().st_size>0
        if not ok: continue
        try: img=Image.open(img_p).convert('RGB')
        except Exception: continue
        W,H=img.size
        for i,line in enumerate(lp.read_text().splitlines()):
            parts=line.split()
            short=len(parts)<5
            if short: continue
            cid=int(float(parts[0]))
            bad=cid not in (0,1)
            if bad: continue
            xc,yc,bw,bh=[float(v) for v in parts[1:5]]
            bw*=(1+CROP_MARGIN); bh*=(1+CROP_MARGIN)
            x0=max(0,(xc-bw/2)*W); y0=max(0,(yc-bh/2)*H); x1=min(W,(xc+bw/2)*W); y1=min(H,(yc+bh/2)*H)
            small=(x1<=x0) or (y1<=y0)
            if small: continue
            out=CROPS_BASE/split/TARGET_CLASSES[cid]; out.mkdir(parents=True,exist_ok=True)
            img.crop((x0,y0,x1,y1)).save(out/f"{img_p.stem}_{i}.jpg",quality=95); n_saved+=1
print(f"Crop GT: {n_saved} file -> {CROPS_BASE}")
BASE_CROPS=CROPS_BASE; CROPS_DIR=CROPS_BASE
for split in ['train','valid','test']:
    for cls in TARGET_CLASSES:
        pt=CROPS_BASE/split/cls
        cnt=len(list(pt.glob('*.jpg'))) if pt.exists() else 0
        print(f"  {split}/{cls}: {cnt}")


## 7. Verifikasi crop (fallback restore manual)
Biasanya sudah terisi dari Sel 6. Hanya untuk yang me-restore `/content/egg_crops` manual.


In [ ]:
# Fallback bila sel otomatis dilewati (restore manual).
import pandas as pd
need=("BASE_CROPS" not in globals()) or (BASE_CROPS is None) or (not Path(BASE_CROPS).exists())
if need:
    _c=Path(globals().get("CROPS_DIR",V3_CROPS_DEFAULT))
    okc=_c.exists() and (_c/"train").exists()
    okd=V3_CROPS_DEFAULT.exists() and (V3_CROPS_DEFAULT/"train").exists()
    BASE_CROPS=_c if okc else (V3_CROPS_DEFAULT if okd else None)
    print(f"BASE_CROPS fallback: {BASE_CROPS}")
else: print(f"BASE_CROPS dari build: {BASE_CROPS}")
hasbase=BASE_CROPS is not None
if hasbase:
    CROPS_DIR=BASE_CROPS
    print('Isi crop:')
    for split in ["train","valid","test"]:
        for cls in TARGET_CLASSES:
            pt=BASE_CROPS/split/cls
            cnt=len(list(pt.glob("*.*"))) if pt.exists() else 0
            print(f"  base/{split}/{cls}: {cnt}")
else: print("BELUM ADA crop. Jalankan sel 4-6 dulu / isi CUSTOM_DIRS.")


## 5. Tambah dataset sendiri (opsional tapi disarankan untuk hasil optimal)

Cara menambah data sendiri — pilih **salah satu atau gabungan**:

**Opsi 1 — folder ImageFolder** (paling mudah). Struktur yang diterima:

```
my_eggs/                          # taruh di /content/ atau Drive
├── train/fertile/*.jpg           # (a) sudah split -> dipakai apa adanya
│        infertile/*.jpg
├── valid/fertile/*.jpg
├── test/fertile/*.jpg
```
atau flat tanpa split:

```
my_eggs_flat/
├── fertile/*.jpg                 # (b) flat -> otomatis di-split 80/10/10 stratified
└── infertile/*.jpg
```

Nama kelas fleksibel (`fertil/Fertile/fertile`, `infertil/Infertile/infertile`, kapital acak) — otomatis dinormalisasi. Format: `jpg/jpeg/png/bmp/webp`.

**Opsi 2 — file ZIP.** Masukkan path ZIP ke `CUSTOM_ZIPS`, otomatis diekstrak.

Hasil: `MERGED_CROPS_DIR` = data v3 + data sendiri, dengan:

- prefix anti-tabrakan (`v3_…`, `custom0_…`), dedup MD5 (file identik tidak digandakan),
- split group-aware sederhana (file custom yang flat di-split stratified 80/10/10, seed sama),
- `merged_manifest.csv` mencatat asal tiap file (`source`, `orig_path`) untuk audit.

Kalau tidak punya data tambahan, **kosongkan saja** — training tetap jalan memakai data v3 murni.

In [ ]:
import hashlib
import shutil
import zipfile
import random
import numpy as np
import pandas as pd
from collections import defaultdict

# ================== ISI DI SINI ==================
CUSTOM_DIRS = []   # contoh: ["/content/my_eggs", "/content/drive/MyDrive/my_eggs_flat"]
CUSTOM_ZIPS = []   # contoh: ["/content/my_eggs.zip"]
CUSTOM_SPLIT = (0.8, 0.1, 0.1)  # hanya dipakai utk folder FLAT (tanpa train/valid/test)
# ===============================================

CLASS_ALIASES = {
    "fertil": "fertile", "fertile": "fertile",
    "infertil": "infertile", "infertile": "infertile",
}
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

random.seed(SEED); np.random.seed(SEED)


def norm_cls(name: str):
    return CLASS_ALIASES.get(str(name).strip().lower())


def md5_of(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _extract_zips():
    out = []
    for z in CUSTOM_ZIPS:
        zp = Path(z)
        if not zp.exists():
            print(f"  ZIP tidak ada, dilewati: {zp}")
            continue
        dest = Path("/content/_custom_unzip") / zp.stem
        if not dest.exists():
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(dest)
            print(f"  ZIP diekstrak: {zp} -> {dest}")
        else:
            print(f"  ZIP sudah diekstrak: {dest}")
        # root aktual: bila ZIP berisi 1 folder, pakai folder dalamnya
        subs = [d for d in dest.iterdir() if d.is_dir()]
        out.append(subs[0] if len(subs) == 1 and not (dest / "train").exists() else dest)
    return out


def _collect_from_root(root: Path, tag: str):
    """Kembalikan list (kelas_norm, path). Mendukung layout split & flat."""
    items = []
    has_split = any((root / s).exists() for s in ["train", "valid", "test", "val"])
    if has_split:
        for split in ["train", "valid", "test", "val"]:
            sd = root / split
            if not sd.exists():
                continue
            for cls_dir in sd.iterdir():
                if not cls_dir.is_dir():
                    continue
                cn = norm_cls(cls_dir.name)
                if cn is None:
                    print(f"  [{tag}] folder kelas tak dikenal dilewati: {cls_dir}")
                    continue
                for f in cls_dir.iterdir():
                    if f.is_file() and f.suffix.lower() in VALID_EXTS:
                        sp = "valid" if split == "val" else split
                        items.append((cn, f, sp))
        return items  # (cls, path, split)
    # flat: root/fertile/*, root/infertile/*
    flat = []
    for cls_dir in root.iterdir():
        if not cls_dir.is_dir():
            continue
        cn = norm_cls(cls_dir.name)
        if cn is None:
            continue
        for f in cls_dir.rglob("*"):
            if f.is_file() and f.suffix.lower() in VALID_EXTS:
                flat.append((cn, f))
    if flat:
        # stratified 80/10/10 per kelas
        tr_r, va_r, te_r = CUSTOM_SPLIT
        out = []
        by_cls = defaultdict(list)
        for cn, f in flat:
            by_cls[cn].append(f)
        for cn, files in by_cls.items():
            files = sorted(files)
            random.Random(SEED).shuffle(files)
            n = len(files); n_tr = int(n * tr_r); n_va = int(n * va_r)
            for i, f in enumerate(files):
                sp = "train" if i < n_tr else ("valid" if i < n_tr + n_va else "test")
                out.append((cn, f, sp))
        print(f"  [{tag}] flat terdeteksi -> auto-split {CUSTOM_SPLIT} stratified per kelas.")
        return out
    print(f"  [{tag}] TIDAK dikenali (butuh train/ atau folder kelas). Isi: {[p.name for p in root.iterdir()]}")
    return []


def build_merged():
    if MERGED_CROPS_DIR.exists():
        shutil.rmtree(MERGED_CROPS_DIR)
    for sp in ["train", "valid", "test"]:
        for cn in TARGET_CLASSES:
            (MERGED_CROPS_DIR / sp / cn).mkdir(parents=True, exist_ok=True)
    manifest, seen_md5 = [], set()
    n_dup = 0

    def add_file(src: Path, split: str, cls: str, source: str, prefix: str):
        nonlocal n_dup
        try:
            h = md5_of(src)
        except Exception:
            return False
        if h in seen_md5:
            n_dup += 1
            return False
        seen_md5.add(h)
        dst = MERGED_CROPS_DIR / split / cls / f"{prefix}_{src.stem}{src.suffix.lower()}"
        k = 1
        while dst.exists():
            dst = MERGED_CROPS_DIR / split / cls / f"{prefix}_{src.stem}_{k}{src.suffix.lower()}"
            k += 1
        shutil.copy2(src, dst)
        manifest.append({"file": f"{split}/{cls}/{dst.name}", "split": split,
                         "class": cls, "source": source, "orig_path": str(src), "md5": h})
        return True

    # 1) data v3 (basis)
    n_base = 0
    if BASE_CROPS is not None and BASE_CROPS.exists():
        for sp in ["train", "valid", "test"]:
            for cn in TARGET_CLASSES:
                for f in (BASE_CROPS / sp / cn).glob("*"):
                    if f.is_file() and f.suffix.lower() in VALID_EXTS:
                        if add_file(f, sp, cn, "v3_base", "v3"):
                            n_base += 1
    print(f"Basis v3 disalin: {n_base} file.")

    # 2) data sendiri
    roots = [Path(d) for d in CUSTOM_DIRS if Path(d).exists()] + _extract_zips()
    for d in CUSTOM_DIRS:
        if not Path(d).exists():
            print(f"  CUSTOM_DIRS tidak ada, dilewati: {d}")
    n_custom = 0
    for i, root in enumerate(roots):
        items = _collect_from_root(root, tag=f"custom{i}")
        print(f"  custom{i} ({root}): {len(items)} file terbaca.")
        for cn, f, sp in items:
            if add_file(f, sp, cn, f"custom{i}:{root.name}", f"custom{i}"):
                n_custom += 1
    print(f"Data sendiri ditambahkan: {n_custom} file (duplikat MD5 dibuang: {n_dup}).")

    if not manifest:
        raise RuntimeError("MERGED KOSONG — tidak ada file dari v3 maupun custom. Cek path di atas.")
    man = pd.DataFrame(manifest)
    man.to_csv(MERGED_CROPS_DIR / "merged_manifest.csv", index=False)
    print(f"\nMerged dataset: {len(man)} file -> {MERGED_CROPS_DIR}")
    print(man.groupby(["split", "class"]).size().unstack(fill_value=0))
    print(man.groupby("source").size())
    return man


merged_manifest = build_merged()
CROPS_DIR = MERGED_CROPS_DIR  # mulai sel ini, training memakai data gabungan

## 6. Audit dataset gabungan (imbalance + kebocoran sederhana)

Menampilkan rasio fertile/infertile per split dan memeriksa duplikat MD5 lintas split (MD5 sama di train & test = kebocoran).

In [ ]:
print("=" * 60)
print("AUDIT DATASET GABUNGAN")
print("=" * 60)
for sp in ["train", "valid", "test"]:
    f = len(list((CROPS_DIR / sp / "fertile").glob("*.*")))
    inf = len(list((CROPS_DIR / sp / "infertile").glob("*.*")))
    tot = f + inf
    print(f"{sp.upper():<6} -> {tot:>5} | fertile {f:>5} ({f/max(tot,1)*100:4.1f}%) | " f"infertile {inf:>5} | rasio f/i {f/max(inf,1):.2f}")
print("=" * 60)
# duplikat lintas split (seharusnya 0 karena MD5 sudah didedup saat merge)
md5_by_split = {}
for sp in ["train", "valid", "test"]:
    md5_by_split[sp] = set(merged_manifest[merged_manifest["split"] == sp]["md5"])
for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    leak = len(md5_by_split[a] & md5_by_split[b])
    print(f"Duplikat MD5 {a} vs {b}: {leak} (" + ("BOCOR!" if leak else "aman") + ")")
tr = merged_manifest[merged_manifest["split"] == "train"]
imb = len(tr[tr["class"] == "fertile"]) / max(len(tr[tr["class"] == "infertile"]), 1)
if imb < 0.5 or imb > 2.0:
    print(f"\nPERHATIAN: train imbalance (rasio {imb:.2f}). Sampler + class weights sudah aktif — biarkan ON.")
else:
    print(f"\nKeseimbangan train OK (rasio {imb:.2f}).")

## 7. Visualisasi sampel crop (cek label tidak tertukar)

Hijau = fertile (pembuluh darah/embrio), merah = infertile (bening). Termasuk sampel dari data sendiri bila ada.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

random.seed(SEED)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for r, cls in enumerate(TARGET_CLASSES):
    files = list((CROPS_DIR / "train" / cls).glob("*.*"))
    samp = random.sample(files, min(5, len(files))) if files else []
    for c in range(5):
        ax = axes[r, c]
        if c < len(samp):
            ax.imshow(Image.open(samp[c]).convert("RGB"))
            src = ""
            try:
                m = merged_manifest[merged_manifest["file"] == f"train/{cls}/{samp[c].name}"]
                src = m.iloc[0]["source"] if len(m) else ""
            except Exception:
                pass
            ax.set_title(f"{cls}\n{samp[c].name[:20]}\n{src}", fontsize=7)
        ax.axis("off")
plt.suptitle("Sampel train gabungan — atas fertile, bawah infertile", fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Dataset & DataLoader optimal

- Augmentasi kuat (crop/scale, flip, rotasi, color jitter, erasing) khusus train; valid/test deterministik.
- `WeightedRandomSampler` + `class weights` menangani imbalance otomatis.
- Assert urutan kelas `fertile < infertile` (kompatibel v3).

In [ ]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms

def seed_everything(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_everything(SEED)

# --- augmentasi disesuaikan untuk candling: fokus tekstur pembuluh darah, cegah bias warna ---
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(30),
    # Jitter warna kuat agar model kebal terhadap variasi lampu candling (kuning/merah/putih)
    transforms.ColorJitter(brightness=0.3, contrast=0.4, saturation=0.3, hue=0.15),
    # Random Grayscale memaksa model mengenali tekstur & kontras urat pembuluh darah
    transforms.RandomGrayscale(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(CROPS_DIR / "train", transform=train_tf)
valid_ds = datasets.ImageFolder(CROPS_DIR / "valid", transform=eval_tf)
test_ds = datasets.ImageFolder(CROPS_DIR / "test", transform=eval_tf)

assert train_ds.classes == TARGET_CLASSES, f"Urutan kelas salah: {train_ds.classes} vs {TARGET_CLASSES}"
assert valid_ds.classes == TARGET_CLASSES
assert test_ds.classes == TARGET_CLASSES

# --- class weights + sampler ---
targets = np.array(train_ds.targets)
counts = np.bincount(targets, minlength=len(TARGET_CLASSES)).astype(float)
class_weights = torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float)
print(f"Train counts: fertile={int(counts[0])}, infertile={int(counts[1])}")
print(f"Class weights: {class_weights.tolist()}")

sampler = None
if USE_SAMPLER:
    sample_w = class_weights[targets].tolist() if USE_CLASS_WEIGHTS else (1.0 / np.maximum(counts[targets], 1)).tolist()
    sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)
    print("WeightedRandomSampler AKTIF.")

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, shuffle=(sampler is None),
                          num_workers=NUM_WORKERS, generator=g if sampler else None)
valid_loader = DataLoader(valid_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
print(f"Train: {len(train_ds)} | Valid: {len(valid_ds)} | Test: {len(test_ds)}")


## 9. Training MobileNetV2 optimal (2 fase + AMP + warmup + cosine)

1. **Fase 1**: backbone frozen, head (dropout 0.2 + linear) dilatih dulu.
2. **Fase 2**: unfreeze semua, AdamW lr kecil + warmup linear + cosine annealing, grad-clip, early stopping berbasis **F2-fertile** (selaras dengan threshold tuning).

Checkpoint terbaik disimpan ke `/content/mobilenet_egg_best.pt` + backup Drive.

In [ ]:
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import fbeta_score

CLS_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {CLS_DEVICE}")
scaler = torch.amp.GradScaler("cuda") if (AMP and CLS_DEVICE.type == "cuda") else None
print(f"AMP: {'ON' if scaler else 'OFF'}")

classifier = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
in_feat = classifier.classifier[1].in_features
classifier.classifier[1] = nn.Sequential(nn.Dropout(p=DROPOUT_P), nn.Linear(in_feat, len(TARGET_CLASSES)))
classifier = classifier.to(CLS_DEVICE)

ce_weight = class_weights.to(CLS_DEVICE) if USE_CLASS_WEIGHTS else None
criterion = nn.CrossEntropyLoss(weight=ce_weight, label_smoothing=LABEL_SMOOTHING)
print(f"Loss weights: {ce_weight.tolist() if ce_weight is not None else 'none'} | label_smoothing={LABEL_SMOOTHING}")


def _mixup(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, None, None
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def run_epoch(model, loader, optimizer=None, use_mixup=False):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_p, all_t = [], []
    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(CLS_DEVICE), y.to(CLS_DEVICE)
            if is_train:
                optimizer.zero_grad()
            if is_train and use_mixup:
                xm, ya, yb, lam = _mixup(x, y, MIXUP_ALPHA)
                with torch.amp.autocast("cuda", enabled=scaler is not None):
                    out = model(xm)
                    loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb)
            else:
                with torch.amp.autocast("cuda", enabled=scaler is not None):
                    out = model(x)
                    loss = criterion(out, y)
            if is_train:
                if scaler:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
            total_loss += loss.item() * x.size(0)
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            n += x.size(0)
            all_p += pred.detach().cpu().tolist()
            all_t += y.detach().cpu().tolist()
    f2 = float(fbeta_score(all_t, all_p, beta=BETA_FERTILE, average="macro", zero_division=0))
    return total_loss / max(n, 1), correct / max(n, 1), f2


# --- Fase 1: head saja ---
for p in classifier.features.parameters():
    p.requires_grad = False
for p in classifier.classifier.parameters():
    p.requires_grad = True
opt_head = optim.Adam(classifier.classifier.parameters(), lr=LR_HEAD)
print("\n--- Fase 1: head (backbone frozen) ---")
for epoch in range(EPOCHS_HEAD):
    tr_loss, tr_acc, tr_f2 = run_epoch(classifier, train_loader, opt_head)
    va_loss, va_acc, va_f2 = run_epoch(classifier, valid_loader)
    print(f"[head {epoch+1}/{EPOCHS_HEAD}] loss {tr_loss:.4f}/{va_loss:.4f} | acc {tr_acc:.4f}/{va_acc:.4f} | F2 {tr_f2:.4f}/{va_f2:.4f}")

# --- Fase 2: fine-tune penuh + warmup + cosine ---
for p in classifier.parameters():
    p.requires_grad = True
opt_full = optim.AdamW(classifier.parameters(), lr=LR_FT, weight_decay=WEIGHT_DECAY)
sched_cos = optim.lr_scheduler.CosineAnnealingLR(opt_full, T_max=max(EPOCHS_FT - WARMUP_EPOCHS, 1))

CLS_BEST_PATH = Path("/content/mobilenet_egg_best.pt")
best_f2, bad_epochs = 0.0, 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f2": []}
print("\n--- Fase 2: fine-tune full ---")
for epoch in range(EPOCHS_FT):
    # warmup linear
    if epoch < WARMUP_EPOCHS:
        warm_lr = LR_FT * (epoch + 1) / max(WARMUP_EPOCHS, 1)
        for pg in opt_full.param_groups:
            pg["lr"] = warm_lr
    tr_loss, tr_acc, tr_f2 = run_epoch(classifier, train_loader, opt_full, use_mixup=USE_MIXUP)
    va_loss, va_acc, va_f2 = run_epoch(classifier, valid_loader)
    if epoch >= WARMUP_EPOCHS:
        sched_cos.step()
    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss); history["val_acc"].append(va_acc); history["val_f2"].append(va_f2)
    print(f"[ft {epoch+1}/{EPOCHS_FT}] loss {tr_loss:.4f}/{va_loss:.4f} | acc {tr_acc:.4f}/{va_acc:.4f} | F2 {tr_f2:.4f}/{va_f2:.4f} | lr {opt_full.param_groups[0]['lr']:.2e}")
    if va_f2 > best_f2:
        best_f2 = va_f2
        bad_epochs = 0
        torch.save({"state_dict": classifier.state_dict(), "img_size": IMG_SIZE,
                    "classes": TARGET_CLASSES, "val_f2": best_f2, "epoch": epoch}, CLS_BEST_PATH)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping di epoch {epoch+1} (F2 tidak membaik {PATIENCE} epoch).")
            break

ckpt = torch.load(CLS_BEST_PATH, map_location=CLS_DEVICE)
classifier.load_state_dict(ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt)
print(f"\nSelesai. Val F2 terbaik: {best_f2:.4f} | Bobot: {CLS_BEST_PATH}")
CLS_DRIVE_PATH = EXPORT_DIR / "mobilenet_egg_best.pt"
shutil.copy2(CLS_BEST_PATH, CLS_DRIVE_PATH)
print(f"Backup ke: {CLS_DRIVE_PATH}")

## 10. Kurva training

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="valid")
ax[0].set_title("Loss"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["val_acc"], label="valid")
ax[1].set_title("Accuracy"); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[2].plot(history["val_f2"], label="val F2", color="green")
ax[2].set_title(f"Val F2 (beta={BETA_FERTILE})"); ax[2].legend(); ax[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Evaluasi pada test crop (threshold default 0.5)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

def collect_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            with torch.amp.autocast("cuda", enabled=(CLS_DEVICE.type == "cuda")):
                probs = torch.softmax(model(x.to(CLS_DEVICE)), dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

valid_probs, valid_labels = collect_probs(classifier, valid_loader)
test_probs, test_labels = collect_probs(classifier, test_loader)
test_preds = test_probs.argmax(1)
print("=" * 55)
print("  EVALUASI TEST CROP (threshold 0.5)")
print("=" * 55)
print(classification_report(test_labels, test_preds, target_names=TARGET_CLASSES, digits=4))
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES)
plt.xlabel("Prediksi"); plt.ylabel("Ground truth")
plt.title("Confusion matrix — test crop gabungan", fontweight="bold")
plt.tight_layout(); plt.show()

## 12. Threshold tuning (F-beta, fokus recall fertile)

FN pada fertile lebih mahal (embrio ikut dibuang) → default `BETA_FERTILE=2.0`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

def fbeta(p, r, beta):
    b2 = beta ** 2
    return (1 + b2) * p * r / max(b2 * p + r, 1e-9)

def metrics_at_thr(probs, labels, thr):
    preds = np.where(probs[:, 0] >= thr, 0, 1)
    tp0 = ((preds == 0) & (labels == 0)).sum()
    fp0 = ((preds == 0) & (labels == 1)).sum()
    fn0 = ((preds == 1) & (labels == 0)).sum()
    return tp0 / max(tp0 + fp0, 1e-9), tp0 / max(tp0 + fn0, 1e-9)

grid = np.arange(0.05, 0.96, 0.02)
scores = [fbeta(*metrics_at_thr(valid_probs, valid_labels, t), BETA_FERTILE) for t in grid]
best_thr_fertile = float(grid[int(np.argmax(scores))])
print(f"Threshold optimal fertile: {best_thr_fertile:.2f} (F{BETA_FERTILE:.0f}={max(scores):.4f})")
plt.figure(figsize=(7, 4.5))
plt.plot(grid, scores, linewidth=2)
plt.axvline(best_thr_fertile, color="red", linestyle="--", label="optimal")
plt.axvline(0.5, color="gray", linestyle=":", label="default")
plt.xlabel("Threshold fertile"); plt.ylabel(f"F{BETA_FERTILE:.0f}")
plt.title("Threshold tuning (valid gabungan)", fontweight="bold")
plt.legend(); plt.grid(alpha=0.3); plt.show()
test_preds_tuned = np.where(test_probs[:, 0] >= best_thr_fertile, 0, 1)
print("\nSetelah tuning:")
print(classification_report(test_labels, test_preds_tuned, target_names=TARGET_CLASSES, digits=4))

## 13. Uji visual prediksi (sampel test gabungan)

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
random.seed(SEED)
all_files = [(0, f) for f in (CROPS_DIR / "test" / "fertile").glob("*.*")]
all_files += [(1, f) for f in (CROPS_DIR / "test" / "infertile").glob("*.*")]
samp = random.sample(all_files, min(8, len(all_files)))
classifier.eval()
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (true_id, fp) in zip(axes.flat, samp):
    x = eval_tf(Image.open(fp).convert("RGB")).unsqueeze(0).to(CLS_DEVICE)
    with torch.no_grad():
        prob = torch.softmax(classifier(x), dim=1).cpu().numpy()[0]
    pred_id = 0 if prob[0] >= best_thr_fertile else 1
    color = "green" if pred_id == true_id else "red"
    ax.imshow(Image.open(fp).convert("RGB"))
    ax.set_title(f"GT:{TARGET_CLASSES[true_id]} Pred:{TARGET_CLASSES[pred_id]}\n" f"p={prob[pred_id]:.2f}", color=color, fontsize=9)
    ax.axis("off")
plt.suptitle("Prediksi sampel test (hijau=benar, merah=salah)", fontweight="bold")
plt.tight_layout(); plt.show()

## 14. Export ONNX + INT8 + config inferensi

Untuk Raspberry Pi / VPS CPU-only via `onnxruntime`. Config ikut menyimpan threshold, daftar sumber data, dan statistik training.

In [ ]:
import json as json_lib
from datetime import datetime
classifier.eval().cpu()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
cls_onnx_path = EXPORT_DIR / "classifier.onnx"
torch.onnx.export(classifier, dummy, str(cls_onnx_path), input_names=["input"], output_names=["logits"], dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=12)
print(f"ONNX: {cls_onnx_path}")
classifier.to(CLS_DEVICE)
try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    cls_int8_path = EXPORT_DIR / "classifier_int8.onnx"
    quantize_dynamic(str(cls_onnx_path), str(cls_int8_path), weight_type=QuantType.QInt8)
    print(f"INT8 : {cls_int8_path}")
except Exception as e:
    print(f"Kuantisasi dilewati ({e}). Install: pip install onnxruntime")
    cls_int8_path = None
shutil.copy2(MERGED_CROPS_DIR / "merged_manifest.csv", EXPORT_DIR / "merged_manifest.csv")
config = {"model": "mobilenet_v2", "classifier": {"file": (cls_int8_path.name if cls_int8_path else cls_onnx_path.name), "file_fp32": cls_onnx_path.name, "imgsz": IMG_SIZE, "classes": TARGET_CLASSES, "threshold_fertile": float(best_thr_fertile), "normalize_mean": [0.485, 0.456, 0.406], "normalize_std": [0.229, 0.224, 0.225]}, "data_sources": merged_manifest.groupby("source").size().to_dict(), "train_counts": {"fertile": int(counts[0]), "infertile": int(counts[1])}, "best_val_f2": float(best_f2), "exported_at": datetime.now().isoformat(timespec="seconds")}
(EXPORT_DIR / "inference_config.json").write_text(json_lib.dumps(config, indent=2))
print(f"\nIsi {EXPORT_DIR}:")
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"  - {f.name} ({f.stat().st_size/1e6:.1f} MB)")

## 15. Contoh inferensi (PyTorch & ONNX)

Verifikasi di Colab; salin folder `models_exported/` ke Raspberry Pi untuk inferensi CPU (butuh `onnxruntime`, `pillow`, `numpy`).

In [ ]:
import json
import numpy as np
from PIL import Image
def predict_pt(crop_path, thr=None):
    import torch.nn as nn
    from torchvision import models, transforms
    thr = best_thr_fertile if thr is None else thr
    m = models.mobilenet_v2()
    m.classifier[1] = nn.Sequential(nn.Dropout(p=DROPOUT_P), nn.Linear(m.classifier[1][0].in_features if isinstance(m.classifier[1], nn.Sequential) else 1280, 2))
    ck = torch.load(CLS_BEST_PATH, map_location="cpu")
    sd = ck["state_dict"] if isinstance(ck, dict) and "state_dict" in ck else ck
    # fallback bila arsitektur head sedikit berbeda (Sequential vs Linear)
    try:
        m.load_state_dict(sd)
    except Exception:
        m.classifier[1] = nn.Linear(1280, 2)
        m.load_state_dict(sd, strict=False)
    m.eval()
    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    x = tf(Image.open(crop_path).convert("RGB")).unsqueeze(0)
    with torch.no_grad():
        prob = torch.softmax(m(x), dim=1).numpy()[0]
    cid = 0 if prob[0] >= thr else 1
    return TARGET_CLASSES[cid], float(prob[cid]), prob.tolist()
def predict_onnx(crop_path, thr=None):
    import onnxruntime as ort
    thr = best_thr_fertile if thr is None else thr
    cfg = json.loads((EXPORT_DIR / "inference_config.json").read_text())["classifier"]
    sess = ort.InferenceSession(str(EXPORT_DIR / cfg["file"]), providers=["CPUExecutionProvider"])
    img = Image.open(crop_path).convert("RGB").resize((cfg["imgsz"], cfg["imgsz"]))
    arr = np.array(img).astype(np.float32) / 255.0
    arr = (arr - np.array(cfg["normalize_mean"])) / np.array(cfg["normalize_std"])
    arr = arr.transpose(2, 0, 1)[None, ...].astype(np.float32)
    logits = sess.run(None, {sess.get_inputs()[0].name: arr})[0][0]
    e = np.exp(logits - logits.max()); probs = (e / e.sum()).tolist()
    cid = 0 if probs[0] >= thr else 1
    return cfg["classes"][cid], float(probs[cid]), probs
demo = list((CROPS_DIR / "test" / "fertile").glob("*.*"))
if demo:
    print("PyTorch:", predict_pt(demo[0]))
    print("ONNX   :", predict_onnx(demo[0]))
else:
    print("Tidak ada file demo di test/fertile.")